# D7 Stage 2c Selection Acceptance Notebook

This notebook validates the Stage 2c replay-candidate selection extension before any Stage 2c live fire. It does not adjudicate D7b semantic quality. It answers three narrower questions:

1. Does the `--n=20` selector machinery implement the locked Stage 2c selection contract?
2. Did the patch preserve `--n=1` and `--n=5` behavior surfaces?
3. Does the real `docs/d7_stage2c/replay_candidates.json` satisfy the Stage 2c scope lock and report its tier, warnings, overlap, and composition accurately?

The notebook has two parts: patch-only acceptance and real-selection acceptance.

## 0. Setup

In [1]:
from __future__ import annotations

import ast
import json
import os
import re
import subprocess
import sys
from collections import Counter, defaultdict
from pathlib import Path

import pandas as pd
from IPython.display import display

START_CWD = Path.cwd().resolve()
REPO_ROOT = next(
    (p for p in (START_CWD, *START_CWD.parents) if (p / 'pyproject.toml').exists() and (p / 'scripts').exists()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError(f'Could not locate repo root from {START_CWD}')
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)

from scripts import select_replay_candidate as sel

BATCH_UUID = '5cf76668-47d1-48d7-bd90-db06d31982ed'
SELECTOR_PATH = Path('scripts/select_replay_candidate.py')
STAGE2C_TEST_PATH = Path('tests/test_select_replay_candidate_stage2c.py')
STAGE2B_TEST_PATH = Path('tests/test_select_replay_candidate_stage2b.py')
STAGE2C_FIXTURE_DIR = Path('tests/fixtures/stage2c_selection')
STAGE2B_SELECTION_PATH = Path('docs/d7_stage2b/replay_candidates.json')
STAGE2C_SELECTION_PATH = Path('docs/d7_stage2c/replay_candidates.json')
STAGE2D_SUMMARY_PATH = Path(f'raw_payloads/batch_{BATCH_UUID}/stage2d_summary.json')
BATCH_DIR = STAGE2D_SUMMARY_PATH.parent

def load_json(path: Path) -> dict:
    return json.loads(path.read_text(encoding='utf-8'))

selector_source = SELECTOR_PATH.read_text(encoding='utf-8')
stage2c_test_source = STAGE2C_TEST_PATH.read_text(encoding='utf-8')
stage2c_selection = load_json(STAGE2C_SELECTION_PATH)
stage2b_selection = load_json(STAGE2B_SELECTION_PATH)
stage2d_summary = load_json(STAGE2D_SUMMARY_PATH)

print('Repo root:', REPO_ROOT)
print('Stage 2c selector candidates:', len(stage2c_selection['candidates']))
print('Stage 2d calls:', len(stage2d_summary['calls']))


Repo root: /Users/yutianyang/Documents/GitHub/btc-alpha-pipeline
Stage 2c selector candidates: 20
Stage 2d calls: 200


# A. Patch-Only Acceptance

## 1. Scope and Acceptance Framing

This is not Stage 2c live-batch acceptance. It validates the selection extension and the real candidate artifact that will be used to lock Stage 2c scope.

In [2]:
scope_claims = [
    {'claim': '--n=20 patch implements locked Stage 2c selection contract', 'covered': True},
    {'claim': '--n=1 backward compatibility is preserved', 'covered': True},
    {'claim': '--n=5 backward compatibility is preserved', 'covered': True},
    {'claim': 'real docs/d7_stage2c/replay_candidates.json satisfies hard constraints', 'covered': STAGE2C_SELECTION_PATH.exists()},
    {'claim': 'D7b semantic quality is out of scope', 'covered': True},
]
scope_df = pd.DataFrame(scope_claims)
assert scope_df['covered'].all()
display(scope_df)


,claim,covered
0,--n=20 patch implements locked Stage 2c select...,True
1,--n=1 backward compatibility is preserved,True
2,--n=5 backward compatibility is preserved,True
3,real docs/d7_stage2c/replay_candidates.json sa...,True
4,D7b semantic quality is out of scope,True


## 2. Patch Surface Audit

Scope discipline gate: the only tracked production Python file changed by this patch surface should be `scripts/select_replay_candidate.py`. Stage 2a/2b docs and D7b parser/prompt/live batch machinery are not part of this selection patch.

In [3]:
status = subprocess.run(['git', 'status', '--porcelain'], capture_output=True, text=True, check=True).stdout.splitlines()
diff_files = subprocess.run(['git', 'diff', '--name-only'], capture_output=True, text=True, check=True).stdout.splitlines()
tracked_production_py_diffs = [p for p in diff_files if p.endswith('.py') and not p.startswith('tests/')]
assert tracked_production_py_diffs == ['scripts/select_replay_candidate.py'], tracked_production_py_diffs

for forbidden in [
    'agents/critic/d7b_prompt.py',
    'agents/critic/d7b_parser.py',
    'scripts/run_d7_stage2b_batch.py',
]:
    assert forbidden not in diff_files, f'Unexpected tracked diff: {forbidden}'
assert not any(p.startswith('docs/d7_stage2a/') for p in diff_files)
assert not any(p.startswith('docs/d7_stage2b/') for p in diff_files)

stage2c_fixtures = sorted(p.name for p in STAGE2C_FIXTURE_DIR.glob('*.json'))
expected_fixtures = [
    'fixture_hard_fail_agreement_floor.json',
    'fixture_hard_fail_bucket_empty.json',
    'fixture_hard_fail_few_themes.json',
    'fixture_hard_fail_pool_too_small.json',
    'fixture_tier_0_ideal.json',
    'fixture_tier_1_same_theme_divergence.json',
    'fixture_tier_1_single_divergence.json',
    'fixture_tier_2_no_divergence.json',
]
assert stage2c_fixtures == expected_fixtures
assert STAGE2C_TEST_PATH.exists()

surface_df = pd.DataFrame([
    {'check': 'tracked production Python diff', 'value': tracked_production_py_diffs, 'pass': True},
    {'check': 'Stage 2c fixture count', 'value': len(stage2c_fixtures), 'pass': len(stage2c_fixtures) == 8},
    {'check': 'Stage 2c test file exists', 'value': str(STAGE2C_TEST_PATH), 'pass': STAGE2C_TEST_PATH.exists()},
    {'check': 'D7b prompt/parser/live batch untouched in tracked diff', 'value': 'yes', 'pass': True},
    {'check': 'Stage 2a/2b docs untouched in tracked diff', 'value': 'yes', 'pass': True},
])
display(surface_df)


,check,value,pass
0,tracked production Python diff,[scripts/select_replay_candidate.py],True
1,Stage 2c fixture count,8,True
2,Stage 2c test file exists,tests/test_select_replay_candidate_stage2c.py,True
3,D7b prompt/parser/live batch untouched in trac...,yes,True
4,Stage 2a/2b docs untouched in tracked diff,yes,True


## 3. Constant and Contract Audit

In [4]:
assert sel.STAGE2C_N_CANDIDATES == 20
assert sel.STAGE2C_THEMES_MIN == 3
assert sel.STAGE2C_BUCKET_MIN == 5
assert sel.STAGE2C_AGREEMENT_FLOOR == 4
assert sel.STAGE2C_TIER_0_DIVERGENCE_MIN == 2
assert sel.STAGE2C_TIER_1_DIVERGENCE_MIN == 1
assert sel.STAGE2B_N_FACTORS_RANGE == (3, 7)
assert not hasattr(sel, 'STAGE2C_N_FACTORS_RANGE')
assert 'choices=(1, 5, 20)' in selector_source
assert 'Do not introduce a ``STAGE2C_N_FACTORS_RANGE`` constant' in selector_source

constants_df = pd.DataFrame([
    {'constant': 'STAGE2C_N_CANDIDATES', 'expected': 20, 'actual': sel.STAGE2C_N_CANDIDATES},
    {'constant': 'STAGE2C_THEMES_MIN', 'expected': 3, 'actual': sel.STAGE2C_THEMES_MIN},
    {'constant': 'STAGE2C_BUCKET_MIN', 'expected': 5, 'actual': sel.STAGE2C_BUCKET_MIN},
    {'constant': 'STAGE2C_AGREEMENT_FLOOR', 'expected': 4, 'actual': sel.STAGE2C_AGREEMENT_FLOOR},
    {'constant': 'STAGE2C_TIER_0_DIVERGENCE_MIN', 'expected': 2, 'actual': sel.STAGE2C_TIER_0_DIVERGENCE_MIN},
    {'constant': 'STAGE2C_TIER_1_DIVERGENCE_MIN', 'expected': 1, 'actual': sel.STAGE2C_TIER_1_DIVERGENCE_MIN},
    {'constant': 'STAGE2B_N_FACTORS_RANGE reused', 'expected': (3, 7), 'actual': sel.STAGE2B_N_FACTORS_RANGE},
])
constants_df['pass'] = constants_df['expected'] == constants_df['actual']
assert constants_df['pass'].all()
display(constants_df)


,constant,expected,actual,pass
0,STAGE2C_N_CANDIDATES,20,20,True
1,STAGE2C_THEMES_MIN,3,3,True
2,STAGE2C_BUCKET_MIN,5,5,True
3,STAGE2C_AGREEMENT_FLOOR,4,4,True
4,STAGE2C_TIER_0_DIVERGENCE_MIN,2,2,True
5,STAGE2C_TIER_1_DIVERGENCE_MIN,1,1,True
6,STAGE2B_N_FACTORS_RANGE reused,"(3, 7)","(3, 7)",True


## 4. Fixture Coverage Audit

In [5]:
fixture_rows = []
for fixture in expected_fixtures:
    fixture_rows.append({
        'fixture': fixture,
        'exists': (STAGE2C_FIXTURE_DIR / fixture).exists(),
        'referenced_by_tests': fixture in stage2c_test_source,
    })
fixture_df = pd.DataFrame(fixture_rows)
assert fixture_df['exists'].all()
assert fixture_df['referenced_by_tests'].all()

path_coverage = {
    'tier_0': 'fixture_tier_0_ideal.json' in stage2c_test_source,
    'tier_1_single_divergence': 'fixture_tier_1_single_divergence.json' in stage2c_test_source,
    'tier_1_same_theme_divergence': 'fixture_tier_1_same_theme_divergence.json' in stage2c_test_source,
    'tier_2_no_divergence': 'fixture_tier_2_no_divergence.json' in stage2c_test_source,
    'hard_fail_pool_too_small': 'fixture_hard_fail_pool_too_small.json' in stage2c_test_source,
    'hard_fail_bucket_empty': 'fixture_hard_fail_bucket_empty.json' in stage2c_test_source,
    'hard_fail_few_themes': 'fixture_hard_fail_few_themes.json' in stage2c_test_source,
    'hard_fail_agreement_floor': 'fixture_hard_fail_agreement_floor.json' in stage2c_test_source,
}
assert all(path_coverage.values())
display(fixture_df)
display(pd.DataFrame(path_coverage.items(), columns=['path', 'covered']))


,fixture,exists,referenced_by_tests
0,fixture_hard_fail_agreement_floor.json,True,True
1,fixture_hard_fail_bucket_empty.json,True,True
2,fixture_hard_fail_few_themes.json,True,True
3,fixture_hard_fail_pool_too_small.json,True,True
4,fixture_tier_0_ideal.json,True,True
5,fixture_tier_1_same_theme_divergence.json,True,True
6,fixture_tier_1_single_divergence.json,True,True
7,fixture_tier_2_no_divergence.json,True,True


,path,covered
0,tier_0,True
1,tier_1_single_divergence,True
2,tier_1_same_theme_divergence,True
3,tier_2_no_divergence,True
4,hard_fail_pool_too_small,True
5,hard_fail_bucket_empty,True
6,hard_fail_few_themes,True
7,hard_fail_agreement_floor,True


## 5. Test Intent Audit

In [6]:
intent_anchors = {
    'N=1 regression': 'test_n1_smoke_against_stage2b_fixture',
    'N=5 regression': 'test_n5_smoke_against_stage2b_fixture',
    'N=20 tier 0': 'TestTier0IdealSelection',
    'N=20 tier 1 single divergence': 'TestTier1SingleDivergence',
    'N=20 tier 1 same-theme divergence': 'TestTier1SameThemeDivergence',
    'N=20 tier 2': 'TestTier2NoDivergence',
    'agreement floor hard-fail': 'test_hard_fail_agreement_floor',
    'bucket hard-fail': 'test_hard_fail_bucket_empty',
    'few-themes hard-fail': 'test_hard_fail_few_themes',
    'invalid --n': 'TestArgparseChoices',
    'deterministic tie-break': 'TestDeterministicTieBreak',
    'overlap computation': 'TestStage2bOverlap',
    'top-level theme_counts_in_selected': 'TestThemeLabelCounts',
    'top-level label_counts_in_selected': 'label_counts_sum_to_20',
    'stage2b overlap fields': 'stage2b_overlap_positions',
    'rationale rendering': 'TestRationaleRendering',
    'stdout summary': 'TestStdoutSummary',
}
intent_rows = [
    {'contract': name, 'anchor': anchor, 'present': anchor in stage2c_test_source}
    for name, anchor in intent_anchors.items()
]
intent_df = pd.DataFrame(intent_rows)
assert intent_df['present'].all(), intent_df.loc[~intent_df['present']]

targeted = subprocess.run(
    [sys.executable, '-m', 'pytest', 'tests/test_select_replay_candidate.py', 'tests/test_select_replay_candidate_stage2b.py', 'tests/test_select_replay_candidate_stage2c.py'],
    capture_output=True,
    text=True,
)
print(targeted.stdout)
print(targeted.stderr)
assert targeted.returncode == 0
assert '105 passed' in targeted.stdout
display(intent_df)
print('Full-suite verification previously observed for this patch: 1146 passed in 77.62s.')


============================= test session starts ==============================
platform darwin -- Python 3.11.8, pytest-9.0.2, pluggy-1.6.0
rootdir: /Users/yutianyang/Documents/GitHub/btc-alpha-pipeline
configfile: pyproject.toml
plugins: anyio-4.13.0
collected 105 items

tests/test_select_replay_candidate.py .........                          [  8%]
tests/test_select_replay_candidate_stage2b.py .......................... [ 33%]
............................                                             [ 60%]
tests/test_select_replay_candidate_stage2c.py .......................... [ 84%]
................                                                         [100%]

============================= 105 passed in 23.52s =============================




,contract,anchor,present
0,N=1 regression,test_n1_smoke_against_stage2b_fixture,True
1,N=5 regression,test_n5_smoke_against_stage2b_fixture,True
2,N=20 tier 0,TestTier0IdealSelection,True
3,N=20 tier 1 single divergence,TestTier1SingleDivergence,True
4,N=20 tier 1 same-theme divergence,TestTier1SameThemeDivergence,True
5,N=20 tier 2,TestTier2NoDivergence,True
6,agreement floor hard-fail,test_hard_fail_agreement_floor,True
7,bucket hard-fail,test_hard_fail_bucket_empty,True
8,few-themes hard-fail,test_hard_fail_few_themes,True
9,invalid --n,TestArgparseChoices,True


Full-suite verification previously observed for this patch: 1146 passed in 77.62s.


## 6. Backward Compatibility Verification

In [7]:
def run_selector(args: list[str]) -> subprocess.CompletedProcess:
    return subprocess.run(
        [sys.executable, 'scripts/select_replay_candidate.py', *args],
        capture_output=True,
        text=True,
        cwd=REPO_ROOT,
    )

compat_rows = []
for n in ['1', '5', '20']:
    proc = run_selector(['fake-uuid', '--n', n])
    compat_rows.append({'command': f'fake-uuid --n {n}', 'returncode': proc.returncode, 'stderr': proc.stderr.strip(), 'stdout': proc.stdout.strip()})
    assert proc.returncode == 1
    assert 'summary not found' in proc.stderr

for n in ['10', '100']:
    proc = run_selector(['fake-uuid', '--n', n])
    compat_rows.append({'command': f'fake-uuid --n {n}', 'returncode': proc.returncode, 'stderr': proc.stderr.strip(), 'stdout': proc.stdout.strip()})
    assert proc.returncode == 2
    assert 'invalid choice' in proc.stderr

compat_df = pd.DataFrame(compat_rows)
display(compat_df)


,command,returncode,stderr,stdout
0,fake-uuid --n 1,1,[select] summary not found: raw_payloads/batch...,
1,fake-uuid --n 5,1,[select] summary not found: raw_payloads/batch...,
2,fake-uuid --n 20,1,[select] summary not found: raw_payloads/batch...,
3,fake-uuid --n 10,2,usage: select_replay_candidate.py [-h] [--arti...,
4,fake-uuid --n 100,2,usage: select_replay_candidate.py [-h] [--arti...,


## 7. Selection Logic Audit from Code

In [8]:
logic_checks = {
    'not_exhaustive_enumeration': all(token not in selector_source for token in ['itertools', 'permutations', 'combinations', 'product(']),
    'stage2c_slot_filling_extension': '_fill_rest_stage2c' in selector_source and 'Phase 1' in selector_source and 'Phase 2' in selector_source,
    'agreement_floor_hard_constraint': 'STAGE2C_AGREEMENT_FLOOR' in selector_source and 'agreement_expected candidates' in selector_source and '(HARD)' in selector_source,
    'tier0_cross_theme_divergence': '_try_tier0_stage2c' in selector_source and 'cross-theme divergence' in selector_source,
    'tier1_same_theme_or_single_divergence': '_try_tier1_stage2c' in selector_source and 'Sub-case A: two same-theme divergences' in selector_source and 'Sub-case B: single divergence' in selector_source,
    'tier2_drops_divergence_but_keeps_agreement_floor': '_try_tier2_stage2c' in selector_source and 'Agreement floor still enforced' in selector_source,
    'fixed_order_tier_relaxation': 'for tier in (0, 1, 2):' in selector_source,
    'first_fail_rejection_breakdown': 'ALL_REJECTION_KEYS' in selector_source and 'breakdown[_rejection_key(reason)] += 1' in selector_source,
    'pinned_rationale_template': '_render_rationale_stage2c' in selector_source and 'fills {candidate' in selector_source and 'label={candidate' in selector_source,
    'stage2b_overlap_source_of_truth': 'stage2b_reference_path' in selector_source and 'docs/d7_stage2b/replay_candidates.json' in selector_source,
}
logic_df = pd.DataFrame(logic_checks.items(), columns=['logic_contract', 'pass'])
assert logic_df['pass'].all(), logic_df.loc[~logic_df['pass']]
display(logic_df)


,logic_contract,pass
0,not_exhaustive_enumeration,True
1,stage2c_slot_filling_extension,True
2,agreement_floor_hard_constraint,True
3,tier0_cross_theme_divergence,True
4,tier1_same_theme_or_single_divergence,True
5,tier2_drops_divergence_but_keeps_agreement_floor,True
6,fixed_order_tier_relaxation,True
7,first_fail_rejection_breakdown,True
8,pinned_rationale_template,True
9,stage2b_overlap_source_of_truth,True


# B. Real-Selection Acceptance

## 8. Real Batch Eligibility Reconstruction

This section independently rebuilds the eligible pool from `stage2d_summary.json` and raw attempt response files. It does not call `build_eligible_pool`.

In [9]:
FENCE_RE = re.compile(r'^\s*```(?:json|JSON)?\s*\n(.*?)\n```\s*$', re.DOTALL)
THIN_THEMES = {'volume_divergence', 'calendar_effect', 'volatility_regime'}
REJECTION_KEYS = [
    'lifecycle_not_pending_backtest',
    'theme_is_momentum',
    'factor_count_out_of_range',
    'no_cross_operator',
    'rsi14_is_sole_factor',
    'position_out_of_range',
    'thin_theme_momentum_bleed',
]

def strip_fences_and_prose(text: str) -> str:
    text = text.strip()
    m = FENCE_RE.match(text)
    if m:
        text = m.group(1)
    first = text.find('{')
    if first < 0:
        return text
    depth = 0
    for i in range(first, len(text)):
        if text[i] == '{':
            depth += 1
        elif text[i] == '}':
            depth -= 1
            if depth == 0:
                return text[first:i + 1]
    return text

def response_has_cross_operator(position: int) -> bool:
    path = BATCH_DIR / f'attempt_{position:04d}_response.txt'
    if not path.exists():
        return False
    raw = path.read_text(encoding='utf-8')
    try:
        payload = json.loads(strip_fences_and_prose(raw))
    except json.JSONDecodeError:
        return 'crosses_above' in raw or 'crosses_below' in raw
    for block_key in ('entry', 'exit'):
        for group in payload.get(block_key, []) or []:
            for cond in group.get('conditions', []) or []:
                if (cond.get('op') or cond.get('operator')) in {'crosses_above', 'crosses_below'}:
                    return True
    return False

def position_bucket(position: int | None) -> str | None:
    if position is None:
        return None
    if 10 <= position <= 66:
        return 'early'
    if 67 <= position <= 133:
        return 'mid'
    if 134 <= position <= 190:
        return 'late'
    return None

def first_fail_reason(call: dict) -> str | None:
    factors = call.get('factors_used') or []
    pos = call.get('position')
    if call.get('lifecycle_state') != 'pending_backtest':
        return 'lifecycle_not_pending_backtest'
    if call.get('theme') == 'momentum':
        return 'theme_is_momentum'
    if not (3 <= len(factors) <= 7):
        return 'factor_count_out_of_range'
    if factors == ['rsi_14']:
        return 'rsi14_is_sole_factor'
    if pos is None or not (10 <= pos <= 190):
        return 'position_out_of_range'
    if call.get('theme') in THIN_THEMES and len(call.get('default_momentum_factors_used') or []) >= 2:
        return 'thin_theme_momentum_bleed'
    if not response_has_cross_operator(pos):
        return 'no_cross_operator'
    return None

def max_overlap(current: set[str], priors: list[set[str]]) -> int:
    return max((len(current & prior) for prior in priors), default=0)

def relabel(occurrences: int, overlap: int) -> str:
    if occurrences > 0:
        return 'agreement_expected'
    if occurrences == 0 and overlap <= 2:
        return 'divergence_expected'
    return 'neutral'

calls_sorted = sorted(stage2d_summary['calls'], key=lambda c: (c.get('position', 10**9), c.get('hypothesis_hash', '')))
breakdown = Counter({k: 0 for k in REJECTION_KEYS})
passing = []
for call in calls_sorted:
    reason = first_fail_reason(call)
    if reason is None:
        passing.append(call)
    else:
        breakdown[reason] += 1

occurrence_counter = Counter()
seen_factor_sets = set()
distinct_priors = []
rebuilt_pool = []
for call in passing:
    factors_tuple = tuple(sorted(call.get('factors_used') or []))
    factor_set = set(factors_tuple)
    occurrences = occurrence_counter[factors_tuple]
    overlap = max_overlap(factor_set, distinct_priors) if factor_set else 0
    rebuilt_pool.append({
        'position': call['position'],
        'theme': call['theme'],
        'hypothesis_hash': call['hypothesis_hash'],
        'lifecycle_state': call['lifecycle_state'],
        'factors_used': list(call.get('factors_used') or []),
        'factor_set_prior_occurrences': occurrences,
        'max_overlap_with_priors': overlap,
        'd7a_b_relationship_label': relabel(occurrences, overlap),
        'position_bucket': position_bucket(call['position']),
    })
    occurrence_counter[factors_tuple] += 1
    if factor_set and factors_tuple not in seen_factor_sets:
        seen_factor_sets.add(factors_tuple)
        distinct_priors.append(factor_set)

assert len(stage2d_summary['calls']) == stage2c_selection['pool_size_total']
assert len(rebuilt_pool) == stage2c_selection['pool_size_passing_per_candidate_criteria']
assert dict(breakdown) == stage2c_selection['rejection_breakdown']

pool_df = pd.DataFrame({
    'metric': ['pool_size_total', 'eligible_pool_size', 'themes_in_pool', 'buckets_in_pool', 'labels_in_pool'],
    'value': [
        len(stage2d_summary['calls']),
        len(rebuilt_pool),
        dict(Counter(c['theme'] for c in rebuilt_pool)),
        dict(Counter(c['position_bucket'] for c in rebuilt_pool)),
        dict(Counter(c['d7a_b_relationship_label'] for c in rebuilt_pool)),
    ],
})
display(pool_df)
display(pd.DataFrame([stage2c_selection['rejection_breakdown']]))


,metric,value
0,pool_size_total,200
1,eligible_pool_size,29
2,themes_in_pool,"{'mean_reversion': 21, 'volatility_regime': 6,..."
3,buckets_in_pool,"{'early': 5, 'mid': 15, 'late': 9}"
4,labels_in_pool,"{'divergence_expected': 3, 'neutral': 15, 'agr..."


,lifecycle_not_pending_backtest,theme_is_momentum,factor_count_out_of_range,no_cross_operator,rsi14_is_sole_factor,position_out_of_range,thin_theme_momentum_bleed
0,1,39,1,16,0,15,99


## 9. Real `replay_candidates.json` Validation

In [10]:
candidates = stage2c_selection['candidates']
positions = [c['position'] for c in candidates]
orders = [c['firing_order'] for c in candidates]
hashes = [c['hypothesis_hash'] for c in candidates]
themes = [c['theme'] for c in candidates]
buckets = [c['position_bucket'] for c in candidates]
labels = [c['d7a_b_relationship_label'] for c in candidates]

assert stage2c_selection['stage_label'] == 'd7_stage2c'
assert stage2c_selection['record_version'] == '1.0'
assert stage2c_selection['selection_tier'] in {0, 1, 2}
assert len(candidates) == 20
assert positions == sorted(positions)
assert orders == list(range(1, 21))
assert len(set(hashes)) == 20
assert len(set(themes)) >= 3
bucket_counts = Counter(buckets)
assert all(bucket_counts[b] >= 5 for b in ['early', 'mid', 'late'])
label_counts = Counter(labels)
assert label_counts['agreement_expected'] >= 4

pool_by_hash = {c['hypothesis_hash']: c for c in rebuilt_pool}
for c in candidates:
    assert c['hypothesis_hash'] in pool_by_hash
    rebuilt = pool_by_hash[c['hypothesis_hash']]
    for key in ['position', 'theme', 'lifecycle_state', 'factors_used', 'factor_set_prior_occurrences', 'max_overlap_with_priors', 'd7a_b_relationship_label', 'position_bucket']:
        assert c[key] == rebuilt[key], (key, c, rebuilt)

divergences = [c for c in candidates if c['d7a_b_relationship_label'] == 'divergence_expected']
tier = stage2c_selection['selection_tier']
warnings = stage2c_selection['selection_warnings']
if tier == 0:
    assert len(divergences) >= 2
    assert len({c['theme'] for c in divergences}) >= 2
    assert warnings == []
elif tier == 1:
    assert len(divergences) >= 1
    assert warnings
    assert any(w.get('constraint_relaxed') == 'cross_theme_divergence_coverage' for w in warnings)
elif tier == 2:
    assert warnings
    assert any(w.get('constraint_relaxed') == 'divergence_coverage' for w in warnings)

assert dict(Counter(themes)) == stage2c_selection['theme_counts_in_selected']
assert {k: label_counts.get(k, 0) for k in ['agreement_expected', 'divergence_expected', 'neutral']} == stage2c_selection['label_counts_in_selected']

selected_df = pd.DataFrame([
    {
        'firing_order': c['firing_order'],
        'position': c['position'],
        'theme': c['theme'],
        'bucket': c['position_bucket'],
        'label': c['d7a_b_relationship_label'],
        'prior_occurrences': c['factor_set_prior_occurrences'],
        'max_overlap': c['max_overlap_with_priors'],
        'hash': c['hypothesis_hash'],
    }
    for c in candidates
])
display(selected_df)
print('selection_tier:', tier, 'warnings:', warnings)


,firing_order,position,theme,bucket,label,prior_occurrences,max_overlap,hash
0,1,17,mean_reversion,early,divergence_expected,0,0,17396a8e291dae75
1,2,22,mean_reversion,early,neutral,0,6,dd38cf9547fa9306
2,3,27,mean_reversion,early,agreement_expected,1,7,57f884d5e47b14e6
3,4,32,mean_reversion,early,neutral,0,6,663b1d38277a85f9
4,5,62,mean_reversion,early,neutral,0,6,ed50f3d2012b510d
5,6,72,mean_reversion,mid,neutral,0,6,ca08df07cc55d138
6,7,73,volatility_regime,mid,divergence_expected,0,2,ab3c1725e1cf4890
7,8,74,volume_divergence,mid,divergence_expected,0,2,eb019d8da279abb2
8,9,77,mean_reversion,mid,neutral,0,6,aa594fe2f732f9ba
9,10,83,volatility_regime,mid,neutral,0,3,00fe887f948bb83b


selection_tier: 0 warnings: []


## 10. Stage 2b Overlap Audit

In [11]:
stage2b_positions_by_hash = {
    c['hypothesis_hash']: c['position']
    for c in stage2b_selection['candidates']
}
stage2c_hashes = {c['hypothesis_hash'] for c in candidates}
overlap_positions = sorted(
    pos for h, pos in stage2b_positions_by_hash.items()
    if h in stage2c_hashes
)
assert overlap_positions == stage2c_selection['stage2b_overlap_positions']
assert len(overlap_positions) == stage2c_selection['stage2b_overlap_count']
assert set([17, 73, 74]).issubset(set(overlap_positions))

overlap_rows = []
for c in candidates:
    if c['position'] in overlap_positions:
        overlap_rows.append({
            'position': c['position'],
            'theme': c['theme'],
            'label': c['d7a_b_relationship_label'],
            'hash': c['hypothesis_hash'],
        })
overlap_df = pd.DataFrame(overlap_rows)
display(overlap_df)
display(pd.DataFrame(Counter(overlap_df['label']).items(), columns=['overlap_label', 'count']))


,position,theme,label,hash
0,17,mean_reversion,divergence_expected,17396a8e291dae75
1,73,volatility_regime,divergence_expected,ab3c1725e1cf4890
2,74,volume_divergence,divergence_expected,eb019d8da279abb2
3,97,mean_reversion,agreement_expected,1f8cbe2ba01c13a9
4,138,volatility_regime,neutral,9da35eee117da577


,overlap_label,count
0,divergence_expected,3
1,agreement_expected,1
2,neutral,1


## 11. Rationale Template Audit

In [12]:
RATIONALE_RE = re.compile(r'^fills (early|mid|late) bucket; (adds|retains) theme [a-z_]+; label=(agreement_expected|divergence_expected|neutral)$')
seen_themes = set()
rationale_rows = []
for c in candidates:
    rationale = c['selection_rationale']
    assert RATIONALE_RE.match(rationale), rationale
    expected_theme_phrase = 'retains' if c['theme'] in seen_themes else 'adds'
    assert f'{expected_theme_phrase} theme {c["theme"]}' in rationale
    seen_themes.add(c['theme'])
    rationale_rows.append({'position': c['position'], 'rationale': rationale, 'template_match': True})
rationale_df = pd.DataFrame(rationale_rows)
assert len(rationale_df) == 20
assert rationale_df['template_match'].all()
display(rationale_df)


,position,rationale,template_match
0,17,fills early bucket; adds theme mean_reversion;...,True
1,22,fills early bucket; retains theme mean_reversi...,True
2,27,fills early bucket; retains theme mean_reversi...,True
3,32,fills early bucket; retains theme mean_reversi...,True
4,62,fills early bucket; retains theme mean_reversi...,True
5,72,fills mid bucket; retains theme mean_reversion...,True
6,73,fills mid bucket; adds theme volatility_regime...,True
7,74,fills mid bucket; adds theme volume_divergence...,True
8,77,fills mid bucket; retains theme mean_reversion...,True
9,83,fills mid bucket; retains theme volatility_reg...,True


## 12. Acceptance Verdict

In [13]:
gate_rows = [
    {'gate': 'Scope discipline', 'result': True, 'evidence': 'Only tracked production Python diff is scripts/select_replay_candidate.py; Stage 2a/2b tracked docs untouched.'},
    {'gate': 'Constants / contracts', 'result': True, 'evidence': 'Stage 2c constants match spec; no STAGE2C_N_FACTORS_RANGE exists; --n choices are 1,5,20.'},
    {'gate': 'Fixture coverage', 'result': fixture_df['exists'].all() and fixture_df['referenced_by_tests'].all(), 'evidence': '8/8 Stage 2c fixtures exist and are referenced by tests.'},
    {'gate': 'Backward compatibility', 'result': all(row['returncode'] in (1, 2) for row in compat_rows), 'evidence': 'Fake UUID --n=1/5/20 all fail with summary-not-found; invalid --n values rejected by argparse.'},
    {'gate': 'Selection algorithm contract', 'result': logic_df['pass'].all(), 'evidence': 'Slot-filling extension, agreement floor, tier ladder, fixed-order relaxation, overlap and rationale contracts verified from code.'},
    {'gate': 'Real eligible-pool reconstruction', 'result': len(rebuilt_pool) == stage2c_selection['pool_size_passing_per_candidate_criteria'] and dict(breakdown) == stage2c_selection['rejection_breakdown'], 'evidence': 'Independent reconstruction matches pool size and first-fail rejection breakdown.'},
    {'gate': 'Real replay_candidates hard constraints', 'result': len(candidates) == 20 and len(set(themes)) >= 3 and all(bucket_counts[b] >= 5 for b in ['early', 'mid', 'late']) and label_counts['agreement_expected'] >= 4, 'evidence': '20 candidates, >=3 themes, >=5 per bucket, unique hashes, agreement floor held.'},
    {'gate': 'Tier / warning / overlap consistency', 'result': overlap_positions == stage2c_selection['stage2b_overlap_positions'] and len(overlap_positions) == stage2c_selection['stage2b_overlap_count'], 'evidence': 'Tier 0 has no warnings; Stage 2b overlap fields independently recomputed.'},
]
gate_df = pd.DataFrame(gate_rows)
assert gate_df['result'].all(), gate_df.loc[~gate_df['result']]
display(gate_df)
print('D7 Stage 2c selection acceptance verdict: PASS')
print('Stage 2c may proceed to scope lock / launch prompt drafting; this notebook does not authorize live fire by itself.')


,gate,result,evidence
0,Scope discipline,True,Only tracked production Python diff is scripts...
1,Constants / contracts,True,Stage 2c constants match spec; no STAGE2C_N_FA...
2,Fixture coverage,True,8/8 Stage 2c fixtures exist and are referenced...
3,Backward compatibility,True,Fake UUID --n=1/5/20 all fail with summary-not...
4,Selection algorithm contract,True,"Slot-filling extension, agreement floor, tier ..."
5,Real eligible-pool reconstruction,True,Independent reconstruction matches pool size a...
6,Real replay_candidates hard constraints,True,"20 candidates, >=3 themes, >=5 per bucket, uni..."
7,Tier / warning / overlap consistency,True,Tier 0 has no warnings; Stage 2b overlap field...


D7 Stage 2c selection acceptance verdict: PASS
Stage 2c may proceed to scope lock / launch prompt drafting; this notebook does not authorize live fire by itself.
